In [1]:
import numpy as np
import yfinance as yf

from docplex.mp.model import Model

In [2]:
tickers = ["MSTR", "KO", "V", "NVDA", "CCJ", "ADBE", "JNJ", "JPM"]
assets_data = yf.download(tickers=tickers, period="1y")

[*********************100%***********************]  8 of 8 completed


In [3]:
assets_data

Price            Close                          ...    Volume                     
Ticker            ADBE         CCJ         JNJ  ...      MSTR       NVDA         V
Date                                            ...                               
2025-04-30  374.980011   45.062263  152.072937  ...  13762900  235044600   9526700
2025-05-01  374.630005   45.421558  150.273117  ...  16944100  236121500   5849900
2025-05-02  380.869995   46.918644  151.888092  ...  15948000  190194800   6113700
2025-05-05  381.059998   46.319813  150.798462  ...  14424100  133163200   3865300
2025-05-06  382.980011   48.056431  150.282837  ...   9695600  158525600   3438500
...                ...         ...         ...  ...       ...        ...       ...
2026-04-23  238.979996  123.849998  230.649994  ...  19559000  113561800   7275000
2026-04-24  245.440002  122.150002  227.500000  ...  14119400  214134400   5616900
2026-04-27  239.309998  123.110001  225.339996  ...  13295400  187172400   6261500
2026-04-28  243.199997  116.309998  227.789993  ...  13858700  180275400   8839900
2026-04-29  243.570007  114.290001  227.350006  ...  14919141  114136225  16641544

[251 rows x 40 columns]

In [4]:
assets_close_price = assets_data["Close"]

In [5]:
assets_close_price

Ticker,ADBE,CCJ,JNJ,JPM,KO,MSTR,NVDA,V
Date,,,,,,,,
2025-04-30,374.980011,45.062263,152.072937,239.953369,70.506119,380.109985,108.894333,342.933472
2025-05-01,374.630005,45.421558,150.273117,242.180084,69.281616,381.600006,111.583702,339.906128
2025-05-02,380.869995,46.918644,151.888092,247.692871,69.631485,394.369995,114.473030,345.017853
2025-05-05,381.059998,46.319813,150.798462,247.741913,69.680069,386.529999,113.793190,346.050140
2025-05-06,382.980011,48.056431,150.282837,244.495056,69.699501,385.600006,113.513260,345.117157
...,...,...,...,...,...,...,...,...
2026-04-23,238.979996,123.849998,230.649994,311.690002,76.279999,172.470001,199.639999,308.880005
2026-04-24,245.440002,122.150002,227.500000,308.279999,76.629997,171.020004,208.270004,309.420013
2026-04-27,239.309998,123.110001,225.339996,311.630005,75.440002,169.199997,216.610001,309.649994


In [6]:
assets_pct_change = assets_close_price.pct_change().dropna()

In [7]:
assets_pct_change

Ticker,ADBE,CCJ,JNJ,JPM,KO,MSTR,NVDA,V
Date,,,,,,,,
2025-05-01,-0.000933,0.007973,-0.011835,0.009280,-0.017367,0.003920,0.024697,-0.008828
2025-05-02,0.016656,0.032960,0.010747,0.022763,0.005050,0.033464,0.025894,0.015039
2025-05-05,0.000499,-0.012763,-0.007174,0.000198,0.000698,-0.019880,-0.005939,0.002992
2025-05-06,0.005039,0.037492,-0.003419,-0.013106,0.000279,-0.002406,-0.002460,-0.002696
2025-05-07,0.005196,0.033022,0.018321,0.000562,0.009481,0.017842,0.031002,0.006183
...,...,...,...,...,...,...,...,...
2026-04-23,-0.066266,-0.020716,0.020124,-0.004249,0.022109,-0.038414,-0.014123,-0.007742
2026-04-24,0.027032,-0.013726,-0.013657,-0.010940,0.004588,-0.008407,0.043228,0.001748
2026-04-27,-0.024976,0.007859,-0.009495,0.010867,-0.015529,-0.010642,0.040044,0.000743


In [8]:
covariance_annualized = assets_pct_change.cov()*np.sqrt(252)
returns_annualized = assets_pct_change.mean()*252

In [9]:
covariance_annualized

Ticker,ADBE,CCJ,JNJ,JPM,KO,MSTR,NVDA,V
Ticker,,,,,,,,
ADBE,0.006216,-0.000227,-0.000298,0.000514,-0.000038,0.001390,0.000124,0.001554
CCJ,-0.000227,0.018429,-0.000346,0.001995,-0.000673,0.005074,0.004814,0.000383
JNJ,-0.000298,-0.000346,0.001792,-0.000067,0.000601,-0.000704,-0.000604,0.000284
JPM,0.000514,0.001995,-0.000067,0.002808,-0.000194,0.002167,0.001440,0.001087
KO,-0.000038,-0.000673,0.000601,-0.000194,0.001663,-0.000984,-0.000891,0.000169
MSTR,0.001390,0.005074,-0.000704,0.002167,-0.000984,0.029502,0.005741,0.000200
NVDA,0.000124,0.004814,-0.000604,0.001440,-0.000891,0.005741,0.007137,0.000064
V,0.001554,0.000383,0.000284,0.001087,0.000169,0.000200,0.000064,0.003046


In [10]:
returns_annualized

Ticker
ADBE   -0.384948
CCJ     1.081410
JNJ     0.419755
JPM     0.278143
KO      0.126104
MSTR   -0.651804
NVDA    0.715440
V      -0.000009
dtype: float64

In [13]:
q = 0.5
b = 3
return_risk_free = 0.0375
weights_array = np.array([1/b for _ in range(len(tickers))])

In [15]:
model = Model(name="combinatorial_portfolio_optimization")
x = np.array([model.binary_var(name=f"x[{i}]") for i in range(len(tickers))])
model.minimize(q*((x*weights_array).T@covariance_annualized.values@(x*weights_array))-(1-q)*(returns_annualized.values@x)+return_risk_free)
model.add_constraint(x.sum() == b)
print(model.prettyprint())

// This file has been generated by DOcplex
// model name is: combinatorial_portfolio_optimization
// single vars section
dvar bool x[0];
dvar bool x[1];
dvar bool x[2];
dvar bool x[3];
dvar bool x[4];
dvar bool x[5];
dvar bool x[6];
dvar bool x[7];

minimize
 0.192474 x[0] - 0.540705 x[1] - 0.209878 x[2] - 0.139071 x[3] - 0.063052 x[4]
 + 0.325902 x[5] - 0.357720 x[6] + 0.000005 x[7] [ 0.000345 x[0]^2
 - 0.000025 x[0]*x[1] - 0.000033 x[0]*x[2] + 0.000057 x[0]*x[3]
 - 0.000004 x[0]*x[4] + 0.000154 x[0]*x[5] + 0.000014 x[0]*x[6]
 + 0.000173 x[0]*x[7] + 0.001024 x[1]^2 - 0.000038 x[1]*x[2]
 + 0.000222 x[1]*x[3] - 0.000075 x[1]*x[4] + 0.000564 x[1]*x[5]
 + 0.000535 x[1]*x[6] + 0.000043 x[1]*x[7] + 0.000100 x[2]^2
 - 0.000007 x[2]*x[3] + 0.000067 x[2]*x[4] - 0.000078 x[2]*x[5]
 - 0.000067 x[2]*x[6] + 0.000032 x[2]*x[7] + 0.000156 x[3]^2
 - 0.000022 x[3]*x[4] + 0.000241 x[3]*x[5] + 0.000160 x[3]*x[6]
 + 0.000121 x[3]*x[7] + 0.000092 x[4]^2 - 0.000109 x[4]*x[5]
 - 0.000099 x[4]*x[6] + 0.00001

In [16]:
result = model.solve()

In [ ]:
result.objective_value

-1.068853015218382

In [24]:
result_array = np.array([result.get_value(f"x[{i}]") for i in range(len(tickers))])

In [25]:
result_array

array([0., 1., 1., 0., 0., 0., 1., 0.])

In [26]:
assets_close_price.columns[result_array.astype(bool)]

Index(['CCJ', 'JNJ', 'NVDA'], dtype='str', name='Ticker')